In [1]:
import torch 
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from transformers.image_utils import load_image
from pathlib import Path

c:\Users\Gaurav Lute\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
DEVICE

'cuda'

In [4]:
# Load image 
image = load_image('closing_disclosure.webp')

In [6]:
# Initialize processor and model
processor = AutoProcessor.from_pretrained("ds4sd/SmolDocling-256M-preview")
model = AutoModelForVision2Seq.from_pretrained(
    "ds4sd/SmolDocling-256M-preview",
    torch_dtype=torch.bfloat16,
    #_attn_implementation="flash_attention_2" if DEVICE == "cuda" else "eager",
).to(DEVICE)

In [11]:
from peft import LoraConfig, get_peft_model


In [10]:
#!pip install peft

In [12]:
# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # SAFE default
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [13]:
# Apply LoRA
model = get_peft_model(model, lora_config)

In [14]:
model.print_trainable_parameters()

trainable params: 755,712 || all params: 257,240,640 || trainable%: 0.2938


In [15]:
# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    },
]

In [16]:
# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


In [17]:
# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

In [18]:
# Populate document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
print(doctags)

<doctag><section_header_level_1><loc_27><loc_29><loc_155><loc_44>Closing Disclosure</section_header_level_1>
<text><loc_29><loc_52><loc_100><loc_59>Closing Information</text>
<text><loc_29><loc_61><loc_62><loc_67>Date Issued</text>
<text><loc_86><loc_61><loc_122><loc_67>4/15/2013</text>
<text><loc_29><loc_69><loc_67><loc_75>Closing Date</text>
<text><loc_86><loc_69><loc_122><loc_75>4/15/2013</text>
<text><loc_29><loc_77><loc_84><loc_83>Disbursement Date</text>
<text><loc_86><loc_77><loc_122><loc_83>4/15/2013</text>
<text><loc_29><loc_85><loc_84><loc_91>Settlement Agent</text>
<text><loc_86><loc_85><loc_137><loc_91>Epsilon Title Co.</text>
<text><loc_29><loc_93><loc_47><loc_99>File #</text>
<text><loc_86><loc_93><loc_116><loc_99>12-3456</text>
<text><loc_29><loc_100><loc_56><loc_106>Property</text>
<text><loc_86><loc_100><loc_155><loc_106>456 Somewhere Ave</text>
<text><loc_29><loc_110><loc_58><loc_116>Sale Price</text>
<text><loc_86><loc_110><loc_120><loc_116>$180,000</text>
<text><loc

In [19]:
# create a docling document
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")


In [20]:
print(doc.export_to_markdown())

## Closing Disclosure

Closing Information

Date Issued

4/15/2013

Closing Date

4/15/2013

Disbursement Date

4/15/2013

Settlement Agent

Epsilon Title Co.

File #

12-3456

Property

456 Somewhere Ave

Sale Price

$180,000

Loan Terms

Can this amount increase after closing?

Loan Amount

$162,000

NO

Interest Rate

3.875%

NO

Monthly Principal &amp; Interest

$761.78

NO

See Projected Payments below for your Estimated Total Monthly Payment

Prepayment Penalty

YES

* As high as $3,240 if you pay off the loan during the first 2 years

Balloon Payment

NO

## Projected Payments

Payment Calculation

Years 1-7

Years 8-30

Principal &amp; Interest

$761.78

$761.78

Mortgage Insurance

+

82.35

+

Estimated Escrow

+

206.13

206.13

Amount can increase over time

Estimated Total Monthly Payment

$1,050.26

$967.91

Estimated Taxes, Insurance &amp; Assessments

$356.13 a month

×Property Taxes

YES

Amount can increase over time

See page 4 for details

×Other: Homeowner's Associ